# Inbound Accumulator File Validation

Every morning the TPA drops a pipe-delimited accumulator file on the PBM's SFTP.
Before anything downstream touches it, we need to verify the file is usable.
A bad file that makes it into production corrupts accumulator balances and leads
to incorrect member cost-sharing on claims adjudicated that day.

This notebook runs the standard intake checks against the raw inbound file
(`tpa_accumulator_file_raw`) and the corresponding control record
(`file_control_log`). Any check that fails gets flagged for the analyst
to investigate before the file is promoted to the clean feed table.

In [0]:
%run ./NB_00_pipeline_utilities

# NB00 - Pipeline Utilities
**Eligibility & Accumulator**

In production, a reconciliation pipeline has dozens of checks spread across multiple notebooks.
If one those check fails, say the TPA eligibility table got dropped by an upstream job, you don't
want the entire pipeline to crash and leave you with nothing. You still want the accumulator
reconciliation, the copay audit, and the rollover checks to run and produce findings.

You also need to know exactly what ran, what passed, and what eventually broke.

This notebook provides shared utilities that every downstream notebook imports via
`%run ./NB_00_pipeline_utilities`. It handles three things:

| Utility | What it does |
|---------|-------------|
| Structured logging | JSON audit entries to `pbm_sim.pipeline_audit_log`, queryable by run, notebook, section |
| Error handling decorator | Wraps each check in try/except so one failure doesn't kill the entire pipeline |
| Table validation helpers | Pre-flight checks before a notebook starts its logic (do the table exists? does it has rows? does it have right columns?) |

These functions are used in NB01 through NB07.

## Structured Logging

Every pipeline run gets a unique `run_id` based on the UTC timestamp when the run started.
It ties together all the log entries from a single execution across notebooks.
If the orchestrator kicks off NB01 through NB07 sequentially, they all share the same run_id.

Log entries go to two places:
1. **stdout** as `[PIPELINE_LOG] {...}` - captured by Databricks job logs, useful for real-time monitoring
2. **Delta table** `pbm_sim.pipeline_audit_log` - the permanent audit trail you can query later

Each entry records the notebook, section, outcome (SUCCESS/WARNING/ERROR/SKIPPED),
row count, timing, and a free-text detail field. The detail field is truncated at 2,000
characters to prevent a full stack trace from bloating the table.

## Error Handling Decorator

This is the core safety mechanism. In a reconciliation pipeline, each notebook runs
multiple independent checks. Take NB03 (accumulator reconciliation) - it checks for
deductible overcharges AND OOPM overcharges. If the deductible check throws an exception
(maybe a column was renamed upstream), you still want the OOPM check to run and produce
its findings. Without this decorator, an unhandled exception in the deductible check
would kill the notebook, and you'd lose the OOPM results too.

The decorator does three things on failure:
1. Catches the exception and logs it as ERROR with the full traceback
2. Returns an empty DataFrame with the expected schema so downstream unions don't break
3. Lets the notebook continue to the next section

On success, it logs the section as SUCCESS with the row count and wall-clock time.

**Usage in a downstream notebook:**
```python
@pipeline_section("NB_03", "deductible_overcharge", finding_schema=convergence_schema)
def run_deductible_check():
    # ... compute df ...
    return df

df_result = run_deductible_check()
# If the check fails, df_result is an empty DataFrame with the right columns
# so the union at the end of the notebook still works
```

## Table Validation Helpers (Pre-flight Checks)

Before a notebook starts its actual logic, it should verify that the tables it depends on
exist and have data. Without this, a missing table produces a cryptic *AnalysisException*
50 cells into the notebook, and you spend 10 minutes figuring out that the problem was
simply that NB00 didn't run.

These are the pre-flight checks:
- **The Table exists?** Catches the case where the upstream notebook didn't run or the table got dropped
- **Are the table empty?** The table exists but has zero rows - maybe the data generator ran but the filter
  produced nothing. This is different from "table doesn't exist" and needs a different response
- **Are the required columns present?** Schema drift - a column was renamed or removed upstream. Better to catch
  this up front with a clear message than to get a "Column 'xyz' not found" error mid-analysis

## Pipeline Metadata

Convenience function for embedding run metadata into output tables.
When NB07 unions all findings, each row can carry the run_id so you can
trace any finding back to the exact pipeline execution that produced it.

Pipeline utilities loaded. Run ID: RUN_20261017_191554


In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark.sql("USE pbm_sim")

NB = "NB_01"

# Pre-flight. It makes sure the tables we need actually exist before we start querying them.
# Without this, a missing table produces a cryptic AnalysisException 50 cells later.
for tbl in ["pbm_sim.tpa_accumulator_file_raw", "pbm_sim.file_control_log"]:
    validate_table_exists(tbl, NB)

df_raw = spark.table("pbm_sim.tpa_accumulator_file_raw")
df_control = spark.table("pbm_sim.file_control_log")

log_entry(NB, "load_source_tables", "SUCCESS",
          detail=f"Raw file: {df_raw.count()} records, Control log: {df_control.count()} entries")

print(f"Raw file loaded: {df_raw.count()} records")
print(f"Control log loaded: {df_control.count()} entries")

[PIPELINE_LOG] {"run_id": "RUN_20261017_191554", "timestamp": "2026-10-17T19:15:56.996240+00:00", "notebook": "NB_01", "section": "load_source_tables", "status": "SUCCESS", "row_count": 0, "detail": "Raw file: 45377 records, Control log: 1 entries", "duration_sec": 0.0}
Raw file loaded: 45377 records
Control log loaded: 1 entries


## 1. Record Count vs Control File

The control file is generated by the TPA at the same time as the data file.
It contains the expected record count. If the numbers don't match, something
went wrong during transmission (truncation, extra header/footer rows, or
a partial upload that got retried). A mismatch doesn't necessarily mean
the data is bad, but it means we can't trust that we received everything.

In [0]:
actual_count = df_raw.count()
control_row = df_control.collect()[0]
expected_count = control_row["expected_record_count"]
file_name = control_row["file_name"]
file_date = control_row["file_date"]

count_diff = actual_count - expected_count

print(f"File: {file_name}")
print(f"File date: {file_date}")
print(f"Expected records (per control): {expected_count}")
print(f"Actual records received:        {actual_count}")
print(f"Difference:                     {count_diff:+d}")

if count_diff == 0:
    print("\n>> PASS: Record count matches control file.")
elif count_diff > 0:
    print(f"\n>> FAIL: {count_diff} MORE records than expected.")
    print("   Possible causes: duplicate transmission, appended records from a prior run,")
    print("   or the control file was generated before all records were finalized.")
else:
    print(f"\n>> FAIL: {abs(count_diff)} FEWER records than expected.")
    print("   Possible causes: truncated transfer, timeout during SFTP upload,")
    print("   or records rejected by the TPA's export process.")

File: tpa_accum_20260915.txt
File date: 2026-09-15
Expected records (per control): 45362
Actual records received:        45377
Difference:                     +15

>> FAIL: 15 MORE records than expected.
   Possible causes: duplicate transmission, appended records from a prior run,
   or the control file was generated before all records were finalized.


## 2. Null Member IDs

A record without a member_id cannot be processed. We can't match it to any
enrollment record, can't apply it to the right accumulator balance, and
can't even determine which plan it belongs to. These get quarantined.

In [0]:
df_null_members = df_raw.filter(F.col("member_id").isNull())
null_count = df_null_members.count()

print(f"Records with NULL member_id: {null_count}")

if null_count > 0:
    print(f"\n>> FAIL: {null_count} records have no member_id. Showing details:")
    display(df_null_members)
else:
    print("\n>> PASS: All records have a member_id.")

Records with NULL member_id: 20

>> FAIL: 20 records have no member_id. Showing details:


member_id,plan_id,benefit_year,medical_deductible_applied,medical_oopm_applied,last_claim_date,file_date
null,PLN001,2026,959.49,1732.25,2026-09-14,2026-09-15
null,PLN026,2026,686.25,977.74,2026-09-07,2026-09-15
null,PLN011,2026,405.85,930.2,2026-09-10,2026-09-15
null,PLN012,2026,248.32,1038.23,2026-09-14,2026-09-15
null,PLN019,2026,668.25,353.36,2026-09-10,2026-09-15
null,PLN017,2026,496.23,1089.62,2026-09-11,2026-09-15
null,PLN039,2026,395.88,1807.58,2026-09-02,2026-09-15
null,PLN029,2026,932.54,1582.28,2026-09-01,2026-09-15
null,PLN004,2026,702.32,614.35,2026-09-14,2026-09-15
null,PLN031,2026,177.44,1226.75,2026-09-14,2026-09-15


## 3. Duplicate Records

Exact duplicates (same member_id, plan_id, benefit_year, and dollar amounts)
may come from a file that was partially transmitted and then retried,
or from the TPA's export job running twice. If we load duplicates, the
downstream reconciliation double-counts the medical side and flags
false overcharges.

In [0]:
# Only check rows that have a member_id (nulls were already flagged above)
df_with_id = df_raw.filter(F.col("member_id").isNotNull())

# Identify EXACT duplicates across all columns
window_dedup = Window.partitionBy(
    "member_id", "plan_id", "benefit_year",
    "medical_deductible_applied", "medical_oopm_applied",
    "last_claim_date", "file_date"
)

df_with_rownum = df_with_id.withColumn("row_num", F.row_number().over(window_dedup.orderBy("member_id")))

df_duplicates = df_with_rownum.filter(F.col("row_num") > 1)
dup_count = df_duplicates.count()

print(f"Duplicate records found: {dup_count}")

if dup_count > 0:
    print(f"\n>> FAIL: {dup_count} exact duplicate rows detected.")
    # Show which members are duplicated
    df_dup_members = df_duplicates.select("member_id", "plan_id", "benefit_year").distinct()
    print(f"   Affecting {df_dup_members.count()} distinct members:")
    display(df_dup_members)
else:
    print("\n>> PASS: No exact duplicates found.")


Duplicate records found: 35

>> FAIL: 35 exact duplicate rows detected.
   Affecting 35 distinct members:


member_id,plan_id,benefit_year
MBR1005127,PLN003,2026
MBR1006668,PLN003,2026
MBR1008009,PLN004,2026
MBR1009935,PLN005,2026
MBR1011760,PLN005,2026
MBR1012350,PLN006,2026
MBR1015008,PLN008,2026
MBR1019912,PLN011,2026
MBR1020049,PLN011,2026
MBR1020603,PLN011,2026


## 4. Negative Accumulator Amounts

Deductible and OOPM applied amounts should always be zero or positive.
A negative value typically means a reversal or adjustment was incorrectly
exported as a standalone balance rather than being netted against the
member's total. Alternatively, it could be a data type issue on the TPA side
where a signed integer overflowed.

In [0]:
df_negative = df_with_id.filter(
    (F.col("medical_deductible_applied") < 0) |
    (F.col("medical_oopm_applied") < 0)
)
neg_count = df_negative.count()

print(f"Records with negative amounts: {neg_count}")

if neg_count > 0:
    print(f"\n>> FAIL: {neg_count} records have negative deductible or OOPM values.")
    display(
        df_negative.select(
            "member_id", "plan_id", "benefit_year",
            "medical_deductible_applied", "medical_oopm_applied"
        )
    )
else:
    print("\n>> PASS: All amounts are non-negative.")

Records with negative amounts: 10

>> FAIL: 10 records have negative deductible or OOPM values.


member_id,plan_id,benefit_year,medical_deductible_applied,medical_oopm_applied
MBR1002200,PLN017,2026,-396.22,249.62
MBR1019846,PLN014,2026,-443.98,448.23
MBR1036938,PLN026,2026,-344.12,321.22
MBR1049841,PLN031,2026,-231.3,499.53
MBR1049786,PLN036,2026,-277.51,241.3
MBR1044933,PLN010,2026,-497.82,168.33
MBR1011709,PLN016,2026,-169.44,247.83
MBR1026640,PLN003,2026,-53.07,161.71
MBR1005432,PLN034,2026,-258.08,123.53
MBR1026226,PLN028,2026,-81.55,330.83


## 5. OOPM Less Than Deductible

The out-of-pocket maximum includes the deductible. By definition, OOPM
applied should always be >= deductible applied for any given member.
If OOPM is lower, something is wrong with how the TPA calculated the
running totals.

In [0]:
df_oopm_check = df_with_id.filter(
    (F.col("medical_oopm_applied") < F.col("medical_deductible_applied")) &
    (F.col("medical_deductible_applied") > 0)  # Ignore zeros
)
oopm_inconsistency_count = df_oopm_check.count()

print(f"Records where OOPM < deductible: {oopm_inconsistency_count}")

if oopm_inconsistency_count > 0:
    print(f"\n>> FAIL: {oopm_inconsistency_count} records have OOPM lower than deductible.")
    display(
        df_oopm_check.select(
            "member_id", "plan_id",
            "medical_deductible_applied", "medical_oopm_applied",
            F.round(
                F.col("medical_deductible_applied") - F.col("medical_oopm_applied"), 2
            ).alias("gap")
        ).orderBy(F.col("gap").desc())
    )
else:
    print("\n>> PASS: OOPM >= deductible for all records.")

Records where OOPM < deductible: 0

>> PASS: OOPM >= deductible for all records.


## 6. Members Not Found in Enrollment

Every member_id in the accumulator file should exist in the eligibility records.
An unknown member_id means either the TPA has someone enrolled that we don't,
or there's a member_id formatting mismatch between systems. Either way,
we can't apply these accumulators to anything.

In [0]:
df_pbm_elig = spark.table("pbm_sim.pbm_eligibility")

# Get distinct member_ids from the raw file (excluding nulls)
df_file_members = df_with_id.select("member_id").distinct()
df_elig_members = df_pbm_elig.select("member_id").distinct()

# Left anti join: members in the file that are NOT in PBM eligibility
df_unknown = df_file_members.join(df_elig_members, on="member_id", how="left_anti")
unknown_count = df_unknown.count()

print(f"Member IDs in file but not in PBM eligibility: {unknown_count}")

if unknown_count > 0:
    print(f"\n>> FAIL: {unknown_count} unrecognized member IDs.")
    display(df_unknown)
else:
    print("\n>> PASS: All member IDs in the file exist in PBM eligibility.")

Member IDs in file but not in PBM eligibility: 2

>> FAIL: 2 unrecognized member IDs.


member_id
MBR1049841
MBR1049786


## 7. Benefit Year Sanity Check

The file should only contain records for the current benefit year (or at most
the prior year during the January overlap window). Anything outside that
range is suspicious.

In [0]:
df_by_year = (
    df_with_id
    .groupBy("benefit_year")
    .agg(F.count("*").alias("record_count"))
    .orderBy("benefit_year")
)

print("Benefit year distribution:")
display(df_by_year)

# Flag any year that isn't 2025 or 2026
df_unexpected_years = df_with_id.filter(~F.col("benefit_year").isin(2025, 2026))
unexpected_year_count = df_unexpected_years.count()

if unexpected_year_count > 0:
    print(f"\n>> FAIL: {unexpected_year_count} records with unexpected benefit year.")
else:
    print("\n>> PASS: All records are for expected benefit years.")

Benefit year distribution:


benefit_year,record_count
2026,45357



>> PASS: All records are for expected benefit years.


## 8. Schema Drift Detection

TPA file schemas change without warning. A column gets renamed, a new field
appears, a date field starts arriving as a string in a different format. When
the schema changes and nobody catches it, the downstream joins silently
misalign or the casts produce NULLs where there should be real values. The
reconciliation runs, finds nothing wrong, and everyone thinks things are fine
until a member gets overcharged.

This check compares the inbound file's actual schema against what we expect.
Any new, missing, or type-changed columns get flagged before the data moves
any further.

In [0]:
# In production this would be loaded from a schema registry or a Delta table
# managed by the data engineering team. Changes to the expected schema go
# through a review process before being promoted.
EXPECTED_COLUMNS = {
    "member_id": "string",
    "plan_id": "string",
    "benefit_year": "int",
    "medical_deductible_applied": "double",
    "medical_oopm_applied": "double",
    "last_claim_date": "date",
    "file_date": "date",
}

actual_schema = {field.name: field.dataType.simpleString() for field in df_raw.schema.fields}

new_columns = set(actual_schema.keys()) - set(EXPECTED_COLUMNS.keys())
missing_columns = set(EXPECTED_COLUMNS.keys()) - set(actual_schema.keys())
type_changes = {
    col: (EXPECTED_COLUMNS[col], actual_schema[col])
    for col in set(EXPECTED_COLUMNS.keys()) & set(actual_schema.keys())
    if EXPECTED_COLUMNS[col] != actual_schema[col]
}

schema_drift_found = bool(new_columns or missing_columns or type_changes)

if schema_drift_found:
    print(">> FAIL: Schema drift detected in inbound file.")
    if new_columns:
        print(f"   New columns (not in expected schema): {sorted(new_columns)}")
    if missing_columns:
        print(f"   Missing columns (expected but absent): {sorted(missing_columns)}")
    if type_changes:
        for col, (expected, actual) in sorted(type_changes.items()):
            print(f"   Type change: {col} expected '{expected}', got '{actual}'")
else:
    print(">> PASS: File schema matches expected schema.")

>> PASS: File schema matches expected schema.


## 9. File Freshness SLA

The TPA is contractually obligated to deliver the accumulator file by a specific
time each business day. When a file arrives late, the entire downstream pipeline
runs on stale accumulator balances. Claims adjudicated that day use yesterday's
(or older) medical-side totals, which means the PBM might not know a member
already met their deductible on the medical side.

We check two things here: is the file_date within the expected delivery window
(yesterday or today), and do all records in the file share the same file_date.
Multiple file_dates in a single file suggests someone concatenated transmissions
from different days, which is a different kind of problem.

In [0]:
from datetime import datetime, timedelta

# SLA window: file_date should be yesterday or today.
# The TPA generates the file at end of day and we pick it up the next morning.
# In production this window would come from the TPA contract table.
today = datetime.now().date()
yesterday = today - timedelta(days=1)

# Check how many distinct file_dates exist in the data.
# There should be exactly one. Multiple dates = concatenated files.
df_file_dates = df_raw.select("file_date").distinct()
file_date_count = df_file_dates.count()

if file_date_count > 1:
    print(f">> FAIL: Multiple file_dates found in the same file ({file_date_count} distinct dates).")
    print("   This suggests multiple transmissions were concatenated.")
    display(df_file_dates.orderBy("file_date"))
    freshness_ok = False
elif file_date_count == 1:
    file_date_value = df_file_dates.collect()[0]["file_date"]
    days_old = (today - file_date_value).days if file_date_value else None

    if days_old is not None and days_old <= 1:
        print(f">> PASS: File date ({file_date_value}) is within SLA window.")
        freshness_ok = True
    elif days_old is not None and days_old > 1:
        print(f">> WARN: File date ({file_date_value}) is {days_old} days old.")
        print(f"   Expected: {yesterday} or {today}")
        print(f"   Possible causes: delayed transmission, reprocessed historical file,")
        print(f"   or the TPA's export job is using a stale date reference.")
        freshness_ok = False
    else:
        print(f">> FAIL: Could not parse file_date for freshness check.")
        freshness_ok = False
else:
    print(f">> FAIL: No file_date values found in the file.")
    freshness_ok = False

>> WARN: File date (2026-09-15) is 32 days old.
   Expected: 2026-10-16 or 2026-10-17
   Possible causes: delayed transmission, reprocessed historical file,
   or the TPA's export job is using a stale date reference.


## 10. Resent File Detection

When an SFTP transfer fails partway through, the TPA may retransmit the entire
file. If the original partial file wasn't cleaned up, we end up with records from
both transmissions. This is different from the duplicate check in section 3:
a resend can have slightly different timestamps or sequence numbers even though
the underlying data is the same member balances.

We look at the control log for multiple entries on the same file_date. If the
TPA sent the file more than once, we need to figure out which transmission is
the authoritative one before proceeding.

In [0]:
df_resend_check = (
    df_control
    .groupBy("file_date")
    .agg(F.count("*").alias("transmission_count"))
    .filter(F.col("transmission_count") > 1)
)

resend_count = df_resend_check.count()

if resend_count > 0:
    print(f">> WARN: {resend_count} date(s) with multiple transmissions detected.")
    print("   Review the control log to determine which transmission is authoritative.")
    display(
        df_control
        .join(df_resend_check.select("file_date"), on="file_date")
        .orderBy("file_date", "file_name")
    )
    resend_found = True
else:
    print(">> PASS: No duplicate transmissions detected in control log.")
    resend_found = False

>> PASS: No duplicate transmissions detected in control log.


## Validation Summary

Everything above rolls up into a single pass/fail decision. In production,
a file that fails any critical check gets held in quarantine and the analyst
opens a ticket with the TPA. Non-critical findings (like a handful of
duplicates that can be deduped safely) might still allow the file to proceed
with a logged exception.

In [0]:
# Collect all results into a summary
results = [
    ("Record count vs control", count_diff == 0, f"Difference: {count_diff:+d}"),
    ("Null member_ids", null_count == 0, f"Found: {null_count}"),
    ("Duplicate records", dup_count == 0, f"Found: {dup_count}"),
    ("Negative amounts", neg_count == 0, f"Found: {neg_count}"),
    ("OOPM < Deductible", oopm_inconsistency_count == 0, f"Found: {oopm_inconsistency_count}"),
    ("Unknown member_ids", unknown_count == 0, f"Found: {unknown_count}"),
    ("Unexpected benefit years", unexpected_year_count == 0, f"Found: {unexpected_year_count}"),
    ("Schema drift", not schema_drift_found, "Drift detected" if schema_drift_found else "Schema matches"),
    ("File freshness SLA", freshness_ok, f"File date: {file_date_value}" if file_date_count == 1 else "Multiple/missing dates"),
    ("Resent file detection", not resend_found, f"Resends: {resend_count}" if resend_count > 0 else "No resends"),
]

summary_data = [(name, "PASS" if passed else "FAIL", detail) for name, passed, detail in results]

df_summary = spark.createDataFrame(summary_data, ["check", "result", "detail"])
display(df_summary)

total_checks = len(results)
passed_checks = sum(1 for _, passed, _ in results if passed)
failed_checks = total_checks - passed_checks

print(f"\n{'=' * 50}")
print(f"  {passed_checks}/{total_checks} checks passed")
if failed_checks > 0:
    print(f"  {failed_checks} checks FAILED - file requires analyst review")
    print(f"  Do NOT promote to clean feed until issues are resolved")
else:
    print(f"  All checks passed - file is clear for promotion")
print(f"{'=' * 50}")

check,result,detail
Record count vs control,FAIL,Difference: +15
Null member_ids,FAIL,Found: 20
Duplicate records,FAIL,Found: 35
Negative amounts,FAIL,Found: 10
OOPM < Deductible,PASS,Found: 0
Unknown member_ids,FAIL,Found: 2
Unexpected benefit years,PASS,Found: 0
Schema drift,PASS,Schema matches
File freshness SLA,FAIL,File date: 2026-09-15
Resent file detection,PASS,No resends



  4/10 checks passed
  6 checks FAILED - file requires analyst review
  Do NOT promote to clean feed until issues are resolved


## Next Steps

For a file that passes validation (or has only non-critical findings that
were handled), the clean version would be written to `tpa_accumulator_feed`
after proper deduplication and null removal.

In this simulation, `tpa_accumulator_feed` already contains the clean data
since it was generated separately from the raw file. In production, the
clean table would be the output of this validation process.